In [ ]:
# Allow importing from src
import sys
sys.path.insert(0, '../src/')

# Fix for draw_geometries crashing on Wayland
import os
os.environ["XDG_SESSION_TYPE"] = "x11"

In [ ]:
import open3d as o3d
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid
from PIL import Image

# DTU

In [ ]:
print("Available scans:")
sc_list = [int(p.stem[4:]) for p in Path(f'../data/DTU').iterdir() if p.is_dir()]
sc_list.sort()
print(", ".join([str(s) for s in sc_list]))

## Load and view stl, points, mesh

In [ ]:
SCAN_NUM = 97
POSTFIX = ""

postfix_str = f"_{POSTFIX}" if len(POSTFIX) > 0 else ""

datadir = Path(f"../data/DTU/scan{SCAN_NUM}").resolve()
print(f"{datadir=}")
arr = np.load(datadir / "cameras.npz")

stl = o3d.io.read_point_cloud(datadir / f"stl{SCAN_NUM:03d}_total.ply")
stl = stl.voxel_down_sample(0.005 * arr["scale_mat_0"][0,0])
print(f"{stl=}")

cloud = o3d.io.read_point_cloud(datadir / f"dmn_cloud{postfix_str}.ply")
cloud.paint_uniform_color([0.5, 0.5, 0.5])
print(f"{cloud=}")

mesh = o3d.io.read_triangle_mesh(datadir / f"dmn_mesh{postfix_str}.ply")
mesh.compute_vertex_normals()
mesh.paint_uniform_color([0.5, 0.5, 0.5])
print(f"{mesh=}")

In [ ]:
o3d.visualization.draw_geometries([mesh], lookat=arr['scale_mat_0'][:3, -1].tolist())

## View camera locations and view directions

In [ ]:
arr = np.load(datadir / "cameras.npz")
# arr['scale_mat_0'], arr['scale_mat_inv_0'], arr['world_mat_0'], arr['world_mat_inv_0'], arr['camera_mat_0'], arr['camera_mat_inv_0']

world_mat = arr['world_mat_0']
scale_mat = arr['scale_mat_0']

# This function is borrowed from IDR: https://github.com/lioryariv/idr
def load_K_Rt_from_P(P):
    out = cv2.decomposeProjectionMatrix(P)
    K = out[0]
    R = out[1]
    t = out[2]

    K = K / K[2, 2]
    intrinsics = np.eye(4)
    intrinsics[:3, :3] = K

    pose = np.eye(4, dtype=np.float32)
    pose[:3, :3] = R.transpose()
    pose[:3, 3] = (t[:3] / t[3])[:, 0]

    return intrinsics, pose

pts = np.zeros((len(arr) // 6, 3))
x1 = np.zeros((len(arr) // 6, 3))
y1 = np.zeros((len(arr) // 6, 3))
directions = []
for i in range(len(arr) // 6):
    P = arr[f'world_mat_{i}'] @ arr[f'scale_mat_{i}']
    P = P[:3, :4]
    intrinsics, pose = load_K_Rt_from_P(P)
    # transforming z-forward, y-down to z-backward, y-up
    pose[:3, 1] *= -1
    pose[:3, 2] *= -1
    pts[i] = (arr['scale_mat_0'] @ pose[:4, -1])[:3]

    direction = np.array((0, 0, -1), np.float32) @ pose[:3, :3].T
    direction /= np.linalg.norm(direction)

    x1[i] = (arr['scale_mat_0'] @ (np.array([1,0,0]) @ pose[:4,:3].T * 0.1 + pose[:4, -1]))[:3]
    y1[i] = (arr['scale_mat_0'] @ (np.array([0,1,0]) @ pose[:4,:3].T * 0.1 + pose[:4, -1]))[:3]

    directions.append(direction)

pts

pose_locs = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
pose_locs.normals = o3d.utility.Vector3dVector(directions)
pose_locs.paint_uniform_color([0.2, 0.7, 0.2])

x1_locs = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(x1))
x1_locs.paint_uniform_color([0.7, 0.2, 0.2])

y1_locs = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(y1))
y1_locs.paint_uniform_color([0.2, 0.7, 0.7])

unit_sphere = o3d.geometry.TriangleMesh.create_sphere()
unit_sphere = o3d.geometry.LineSet.create_from_triangle_mesh(unit_sphere)

axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.2)


In [ ]:
o3d.visualization.draw_geometries([mesh, pose_locs, x1_locs, y1_locs], point_show_normal=True, lookat=arr['scale_mat_0'][:3, -1].tolist())

## View images

In [ ]:
imgs = (datadir / "image").glob("[0-9]*.png")
masks = (datadir / "mask").glob("[0-9]*.png")

ls = []
for img, mask in zip(sorted(imgs), sorted(masks)):
    image = np.asarray(Image.open(img))
    alpha = np.asarray(Image.open(mask))
    image = np.concat([image, alpha.mean(-1, keepdims=True)], axis=-1)
        
    ls.append(image / 255)

imgs = np.stack(ls, 0)

fig, ax = plt.subplots(1, 1, figsize=(25, 25))
ax.imshow(make_grid(torch.tensor(imgs).permute(0, 3, 1, 2), 5, padding=10).permute(1, 2, 0))
ax.axis('off')
pass

# NeSy (NeRF Synthetic)

In [ ]:
print("Available scenes:")
sc_list = [p.stem for p in Path(f'../data/NeSy').iterdir() if p.is_dir()]
sc_list.sort()
print(", ".join([str(s) for s in sc_list]))

In [ ]:
SCENE_NAME = "materials"
POSTFIX = ""

postfix_str = f"_{POSTFIX}" if len(POSTFIX) > 0 else ""

datadir = Path(f"../data/NeSy/{SCENE_NAME}").resolve()

cloud = o3d.io.read_point_cloud(datadir / f"dmn_cloud{postfix_str}.ply")
cloud.paint_uniform_color([0.5, 0.5, 0.5])
print(f"{cloud=}")

mesh = o3d.io.read_triangle_mesh(datadir / f"dmn_mesh{postfix_str}.ply")
mesh.compute_vertex_normals()
mesh.paint_uniform_color([0.5, 0.5, 0.5])
print(f"{mesh=}")

model = o3d.io.read_triangle_mesh(datadir / f"model.ply")
model.compute_vertex_normals()
model.paint_uniform_color([0.5, 0.5, 0.5])
print(f"{model=}")

n2m = o3d.io.read_triangle_mesh(datadir / f"nerf2mesh.obj")
n2m.compute_vertex_normals()
n2m.paint_uniform_color([0.5, 0.5, 0.5])
print(f"{n2m=}")

In [ ]:
o3d.visualization.draw_geometries([n2m], lookat=[0, 0.0, 0.1], front=[0.2, -1, 0.5], up=[0, 0, 1], zoom=0.5)